# 🏆 Entrenamiento del Modelo v5 — Maestría con SE-ResNet-8 (Cognitive Curriculum)

Entrenamiento en Google Colab con aceleración GPU (NVIDIA T4 / A100):
- **Arquitectura:** `RedSEResNetAjedrez` con 8 bloques residuales y atención por canales Squeeze-and-Excitation (Hu et al., 2018; He et al., 2016).
- **Capacidad:** 192 canales convolucionales de alta capacidad táctica.
- **Dataset:** Lichess Open Database (2017-02), filtrado a partidas de jugadores maestros con **ELO >= 2000**.
- **Optimizador & Regularización:** AdamW ($lr=10^{-3}$, weight_decay=$10^{-4}$), Cosine Annealing Scheduler y Cross-Entropy Loss con Label Smoothing ($0.05$).
- **Inferencia Táctica & Salida:** Guardado y descarga directa automática (`.pt`) a tu computadora sin requerir espacio en Google Drive.

> **Importante:** Asegúrate de activar GPU antes de iniciar: `Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU (T4)`.

## 1. Clonar el repositorio y configurar el entorno
Clona el código fuente actualizado del proyecto (`backend/` y `training/`) y configura el PYTHONPATH.

In [ ]:
import os
import sys

# 1. Limpieza y clonado fresco del repositorio
%cd /content
!rm -rf /content/ajedrez
!git clone --depth 1 https://github.com/larzekao9/Brazo-Rob-tico-con-Inteligencia-Artificial-para-el-Aprendizaje-del-Ajedrez.git /content/ajedrez
%cd /content/ajedrez

# 2. Agregar directorio al sys.path para importar backend y training
if '/content/ajedrez' not in sys.path:
    sys.path.insert(0, '/content/ajedrez')

print('\n✅ Repositorio clonado exitosamente. Verificando estructura:')
!ls -la backend/servicios/aprendizaje/


## 2. Instalar dependencias del sistema y librerías
Instala `python-chess`, `zstandard` y el motor de evaluación offline `Stockfish`.

In [ ]:
import torch

# 1. Dependencias de Python fijadas
!pip install -q python-chess==1.999.0 zstandard==0.25.0

# 2. Instalación de Stockfish para benchmarking científico
!apt-get update -qq && apt-get install -y -qq stockfish
!ln -sf /usr/games/stockfish /usr/local/bin/stockfish

# 3. Verificación de GPU
dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\n🚀 Dispositivo activo: {dispositivo}')
if torch.cuda.is_available():
    print(f'   GPU detectada: {torch.cuda.get_device_name(0)}')
    print(f'   Memoria VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('⚠️ ADVERTENCIA: No se detectó GPU. Ve a Entorno de ejecución > Cambiar tipo de entorno de ejecución y activa GPU (T4).')


## 3. Descargar el dataset maestro de Lichess
Descarga el archivo de partidas históricas estándar de Lichess (`2017-02.pgn.zst`, ~1.7 GB comprimido).

In [ ]:
!mkdir -p training/data
!wget -q --show-progress -O training/data/lichess_db_standard_rated_2017-02.pgn.zst \
    https://database.lichess.org/standard/lichess_db_standard_rated_2017-02.pgn.zst
!ls -lh training/data/


## 4. Configurar directorio local de Checkpoints
Crea la carpeta `/content/checkpoints/` donde se almacenarán los modelos generados antes de descargarlos a tu PC.

In [ ]:
import os
CARPETA_CHECKPOINTS = '/content/checkpoints'
os.makedirs(CARPETA_CHECKPOINTS, exist_ok=True)
print('📁 Carpeta local de checkpoints lista en:', CARPETA_CHECKPOINTS)


## 5. Extracción y Curación de Datos (Partidas Maestras ELO >= 2000)
Filtra partidas de alta maestría táctica mediante lectura en streaming de `pgn_to_samples`.
Con 12,000 partidas de maestros se obtienen ~600,000 posiciones de alto valor estratégico.

In [ ]:
import sys
if '/content/ajedrez' not in sys.path:
    sys.path.insert(0, '/content/ajedrez')

from training.data_pipeline import pgn_to_samples

# Parámetros del modelo v5 (Maestría)
LIMITE_PARTIDAS = 12000  # 12,000 partidas de maestros dan ~600,000 posiciones
ELO_MINIMO = 2000

print(f'♟️ Extrayendo hasta {LIMITE_PARTIDAS} partidas de maestros con ELO >= {ELO_MINIMO}...')
muestras = pgn_to_samples(
    'training/data/lichess_db_standard_rated_2017-02.pgn.zst',
    limite_partidas=LIMITE_PARTIDAS,
    elo_minimo=ELO_MINIMO,
)
print(f'✅ ¡Extracción completa! {len(muestras):,} muestras procesadas (tensor 8x8x12 -> jugada clase)')


## 6. Preparación de DataLoaders (Train / Validation Split 90/10)
Crea los cargadores por lotes (`batch_size=128`) para optimizar el rendimiento del pipeline en la GPU.

In [ ]:
import random
import sys
if '/content/ajedrez' not in sys.path:
    sys.path.insert(0, '/content/ajedrez')

import torch
from torch.utils.data import DataLoader, Dataset
from backend.servicios.aprendizaje.modelo_jugadas import (
    NUM_CLASES,
    RedSEResNetAjedrez,
    tensor_a_entrada_red,
)

class DatasetJugadas(Dataset):
    def __init__(self, muestras):
        self.muestras = muestras

    def __len__(self):
        return len(self.muestras)

    def __getitem__(self, indice):
        tensor_posicion, etiqueta = self.muestras[indice]
        return tensor_a_entrada_red(tensor_posicion), etiqueta

random.seed(42)
muestras_mezcladas = muestras[:]
random.shuffle(muestras_mezcladas)
corte = int(0.9 * len(muestras_mezcladas))
muestras_train = muestras_mezcladas[:corte]
muestras_val = muestras_mezcladas[corte:]

cargador_train = DataLoader(DatasetJugadas(muestras_train), batch_size=128, shuffle=True, pin_memory=True, num_workers=2)
cargador_val = DataLoader(DatasetJugadas(muestras_val), batch_size=128, pin_memory=True, num_workers=2)

print(f'📊 Conjuntos preparados: Train = {len(muestras_train):,} muestras | Val = {len(muestras_val):,} muestras')


## 7. Entrenamiento de la Red SE-ResNet-8 (v5 Maestría)
Entrena la red neuronal profunda de 8 bloques residuales con Squeeze-and-Excitation, optimizador AdamW, Cosine Annealing y Label Smoothing (0.05).

In [ ]:
import datetime
import torch

dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🚀 Entrenando en dispositivo: {dispositivo}')

# Instanciar arquitectura SE-ResNet v5 (8 bloques residuales con SE, 192 canales)
red = RedSEResNetAjedrez(canales=192, cantidad_bloques=8, cantidad_clases=NUM_CLASES).to(dispositivo)

CANTIDAD_EPOCAS = 25
optimizador = torch.optim.AdamW(red.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizador, T_max=CANTIDAD_EPOCAS, eta_min=1e-5)
funcion_perdida = torch.nn.CrossEntropyLoss(label_smoothing=0.05)

def evaluar(cargador):
    red.eval()
    correctas, total = 0, 0
    with torch.no_grad():
        for entradas, etiquetas in cargador:
            entradas, etiquetas = entradas.to(dispositivo), etiquetas.to(dispositivo)
            predicciones = red(entradas).argmax(dim=1)
            correctas += (predicciones == etiquetas).sum().item()
            total += etiquetas.size(0)
    return correctas / total if total else 0.0

mejor_acc = 0.0
ruta_mejor_modelo = f'{CARPETA_CHECKPOINTS}/modelo_jugadas_v5_mejor.pt'

print(f'🔥 Iniciando entrenamiento de {CANTIDAD_EPOCAS} épocas...')
for epoca in range(1, CANTIDAD_EPOCAS + 1):
    red.train()
    perdida_acumulada = 0.0
    for entradas, etiquetas in cargador_train:
        entradas, etiquetas = entradas.to(dispositivo), etiquetas.to(dispositivo)
        optimizador.zero_grad()
        salida = red(entradas)
        perdida = funcion_perdida(salida, etiquetas)
        perdida.backward()
        optimizador.step()
        perdida_acumulada += perdida.item() * entradas.size(0)

    scheduler.step()
    lr_actual = optimizador.param_groups[0]['lr']
    perdida_promedio = perdida_acumulada / len(muestras_train)
    accuracy_val = evaluar(cargador_val)

    print(f'Época {epoca:02d}/{CANTIDAD_EPOCAS} | LR: {lr_actual:.6f} | Pérdida: {perdida_promedio:.4f} | Accuracy Val: {accuracy_val:.2%}')

    if accuracy_val > mejor_acc:
        mejor_acc = accuracy_val
        torch.save({
            'state_dict': red.state_dict(),
            'num_clases': NUM_CLASES,
            'arquitectura': 'se_resnet',
            'canales': 192,
            'cantidad_bloques': 8,
            'epoca': epoca,
            'accuracy_val': accuracy_val,
            'fecha': datetime.date.today().isoformat(),
        }, ruta_mejor_modelo)
        print(f'  ⭐ ¡Nuevo récord guardado ({accuracy_val:.2%})!')

print(f'\n🏆 Entrenamiento finalizado. Mejor precisión alcanzada en validación: {mejor_acc:.2%}')


## 8. Guardar y Descargar Checkpoint a tu Computadora
Guarda el checkpoint final con todos los metadatos y lo descarga directamente a tu computadora mediante el navegador (`files.download()`).

In [ ]:
from google.colab import files
import datetime
import os
import torch

fecha_hoy = datetime.date.today().isoformat()
nombre_checkpoint = f'modelo_jugadas_v5_{fecha_hoy}.pt'
ruta_checkpoint = f'{CARPETA_CHECKPOINTS}/{nombre_checkpoint}'

# Si hubo mejor modelo guardado, cargar sus pesos óptimos
if os.path.exists(ruta_mejor_modelo):
    mejor_ckpt = torch.load(ruta_mejor_modelo, map_location='cpu')
    red.load_state_dict(mejor_ckpt['state_dict'])
    print(f'📦 Exportando pesos óptimos del modelo (Acc: {mejor_acc:.2%})')

checkpoint_datos = {
    'state_dict': red.state_dict(),
    'num_clases': NUM_CLASES,
    'arquitectura': 'se_resnet',
    'canales': 192,
    'cantidad_bloques': 8,
    'cantidad_partidas': LIMITE_PARTIDAS,
    'elo_minimo': ELO_MINIMO,
    'cantidad_muestras': len(muestras),
    'accuracy_val': mejor_acc if mejor_acc > 0 else accuracy_val,
    'fecha': fecha_hoy,
}

torch.save(checkpoint_datos, ruta_checkpoint)
print(f'✅ Checkpoint v5 guardado exitosamente en: {ruta_checkpoint}')
print('⬇️ Iniciando descarga directa a tu computadora...')
files.download(ruta_checkpoint)


## 9. Evaluación Científica del Modelo (Benchmark contra Stockfish)
Evalúa el modelo v5 contra partidas de test nunca vistas en entrenamiento y mide la calidad de las jugadas según pérdida de centipawns:
- **Top-1 Accuracy:** Coincidencia exacta con la jugada del Gran Maestro.
- **Aceptable:** Pérdida < 50 cp con respecto a Stockfish (jugada sólida).
- **Imprecisión:** Pérdida entre 50 y 100 centipawns.
- **Error:** Pérdida entre 100 y 300 centipawns.
- **Blunder:** Pérdida > 300 centipawns (error táctico grave).

In [ ]:
from training.evaluar_modelo import evaluar_modelo, BALDES_ERROR

print('--- 🔬 Evaluando Modelo v5 (SE-ResNet-8 192C + Partidas Maestras ELO >= 2000) ---')
resultado = evaluar_modelo(
    'training/data/lichess_db_standard_rated_2017-02.pgn.zst',
    cantidad_partidas=20,
    saltar_partidas=35000,  # Partidas no vistas en entrenamiento
    ruta_checkpoint=ruta_checkpoint,
)

print(f'\nTotal de jugadas evaluadas: {resultado.total_jugadas}')
print(f'Accuracy Top-1 (Coincidencia con Maestro): {resultado.accuracy:.2%}')
print('\nDistribución de calidad de jugadas vs Stockfish:')
for balde in BALDES_ERROR:
    porcentaje = (resultado.distribucion_errores[balde] / resultado.total_jugadas) * 100
    print(f'  - {balde.capitalize():12s}: {resultado.distribucion_errores[balde]:4d} ({porcentaje:.1f}%)')


## 10. Demostración de Inferencia Táctica y Explicabilidad
Prueba la función `predecir_jugada_maestra()` desarrollada para el brazo robótico, que combina predicción neuronal pura, poda táctica anti-trampas (< 3 ms) y saliencia visual.

In [ ]:
import time
from backend.servicios.aprendizaje.inferencia import predecir_jugada_maestra, predecir_top_jugadas

# Posición de prueba táctica
fen_test = "r1bqkb1r/pppp1ppp/2n5/4p3/2B1n3/5N2/PPPP1PPP/RNBQK2R w KQkq - 0 5"

t0 = time.perf_counter()
jugada_elegida = predecir_jugada_maestra(fen_test, ruta_checkpoint=ruta_checkpoint)
latencia_ms = (time.perf_counter() - t0) * 1000
top3 = predecir_top_jugadas(fen_test, top_n=3, ruta_checkpoint=ruta_checkpoint)

print("🤖 Demostración de Inferencia Maestra para el Brazo Robótico:")
print(f"  - Posición FEN: {fen_test}")
print(f"  - Jugada Maestra Recomendada: {jugada_elegida}")
print(f"  - Latencia de Inferencia: {latencia_ms:.2f} ms")
print("  - Top 3 jugadas consideradas por la red:")
for jugada, prob in top3:
    print(f"     • {jugada:6s} -> Confianza: {prob*100:.2f}%")
